In [1]:
# Dataset 1: zeroshot/twitter-financial-news-sentiment
# Dataset 2: zeroshot/twitter-financial-news-topic

In [2]:
# Step 1: 安装依赖

!pip install transformers datasets evaluate accelerate -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


In [3]:
# Step 2: 加载数据集

from datasets import load_dataset

dataset = load_dataset("zeroshot/twitter-financial-news-sentiment")
print(dataset)
print(dataset["train"][0])  # 看一下数据结构

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sent_train.csv: 0.00B [00:00, ?B/s]

sent_valid.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/9543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2388 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9543
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2388
    })
})
{'text': '$BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT', 'label': 0}


In [4]:
# Step 3: 对数据集进行采样（加快运行速度）
# 先采样
dataset["train"] = dataset["train"].shuffle(seed=42).select(range(5000))
validation_full = dataset["validation"].shuffle(seed=42)

# 把validation拆成两半
half = len(validation_full) // 2
dataset["validation"] = validation_full.select(range(half))
dataset["test"] = validation_full.select(range(half, len(validation_full)))

print(dataset)
print(f"Train: {len(dataset['train'])}")
print(f"Validation: {len(dataset['validation'])}")
print(f"Test: {len(dataset['test'])}")

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 5000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1194
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1194
    })
})
Train: 5000
Validation: 1194
Test: 1194


In [5]:
# Step 4：加载 Tokenizer 并处理数据

from transformers import AutoTokenizer, DataCollatorWithPadding

model_name = "jtatman/finetuning-twitter-finance-sentiment-distilbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)
    # 注意：去掉 padding=True，改为动态padding

tokenized_dataset = dataset.map(tokenize, batched=True)

# 使用动态padding，让collator在每个batch内自动对齐长度
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized_dataset)

config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1194 [00:00<?, ? examples/s]

Map:   0%|          | 0/1194 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5000
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1194
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1194
    })
})


In [6]:
# Step 5：加载预训练模型

from transformers import AutoModelForSequenceClassification

# 手动定义label映射
id2label = {0: "Bearish", 1: "Bullish", 2: "Neutral"}
label2id = {"Bearish": 0, "Bullish": 1, "Neutral": 2}
num_labels = 3

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

print("Labels:", id2label)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Labels: {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}


In [7]:
# Step 6：设置评估指标

import evaluate
import numpy as np

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [8]:
# Step 7: 训练配置

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./finbert-sentiment-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",        # 新版本改名了
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_dir="./logs",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [9]:
# Step 8：开始训练

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.545058,0.873534


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=157, training_loss=0.16264900887847705, metrics={'train_runtime': 32.9326, 'train_samples_per_second': 151.825, 'train_steps_per_second': 4.767, 'total_flos': 70898230068048.0, 'train_loss': 0.16264900887847705, 'epoch': 1.0})

In [10]:
# Step 9: 在测试集上评估，记录结果

import time

start = time.time()
results = trainer.evaluate(tokenized_dataset["test"])
end = time.time()

print(f"Test Accuracy: {results['eval_accuracy']:.4f}")
print(f"Runtime: {end - start:.2f} seconds")

Test Accuracy: 0.8819
Runtime: 1.77 seconds


In [12]:
from huggingface_hub import login

# 登录
login()

repo_name_sent = "Nora0211/finbert-sentiment-finetuned"

# 保存模型
trainer.save_model("./finbert-sentiment-finetuned")
tokenizer.save_pretrained("./finbert-sentiment-finetuned")

# 重新加载
from transformers import AutoModelForSequenceClassification, AutoTokenizer
model_sent = AutoModelForSequenceClassification.from_pretrained("./finbert-sentiment-finetuned")
tokenizer_sent = AutoTokenizer.from_pretrained("./finbert-sentiment-finetuned")

# 上传
model_sent.push_to_hub(repo_name_sent)
tokenizer_sent.push_to_hub(repo_name_sent)

print("Pipeline 1 模型上传完成")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...5k7kmh_/model.safetensors:  30%|##9       | 80.0MB /  268MB            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Pipeline 1 模型上传完成（旧仓库已删除）


In [13]:
# 加载数据集
from datasets import load_dataset, DatasetDict

dataset2 = load_dataset("zeroshot/twitter-financial-news-topic")

# 从train里划分10%作为validation，原validation作为test
train_full2 = dataset2["train"].shuffle(seed=42).select(range(5000))
split2 = train_full2.train_test_split(test_size=0.1, seed=42)

dataset2 = DatasetDict({
    "train": split2["train"],
    "validation": split2["test"],
    "test": dataset2["validation"].shuffle(seed=42).select(range(1000))
})

print(f"Train: {len(dataset2['train'])}")
print(f"Validation: {len(dataset2['validation'])}")
print(f"Test: {len(dataset2['test'])}")

README.md: 0.00B [00:00, ?B/s]

topic_train.csv: 0.00B [00:00, ?B/s]

topic_valid.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/16990 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4117 [00:00<?, ? examples/s]

Train: 4500
Validation: 500
Test: 1000


In [14]:
# 定义label映射

# 20个金融话题
id2label2 = {
    0: "Analyst Update",
    1: "Fed & Central Banks",
    2: "Company & Product News",
    3: "Treasuries & Corporate Debt",
    4: "Dividend",
    5: "Earnings",
    6: "Energy & Oil",
    7: "Financials",
    8: "Currencies",
    9: "General News & Opinion",
    10: "Gold, Metals & Materials",
    11: "IPO",
    12: "Legal & Regulation",
    13: "M&A & Investments",
    14: "Macro & Economy",
    15: "Markets",
    16: "Politics",
    17: "Personnel Change",
    18: "Stock Commentary",
    19: "Stock Movement"
}

label2id2 = {v: k for k, v in id2label2.items()}
num_labels2 = 20
print("Number of labels:", num_labels2)

Number of labels: 20


In [15]:
# 加载Tokenizer和处理数据

from transformers import AutoTokenizer, DataCollatorWithPadding

model_name2 = "leonas5555/finnews-topic-single-classify"
tokenizer2 = AutoTokenizer.from_pretrained(model_name2)

def tokenize2(batch):
    return tokenizer2(batch["text"], truncation=True, max_length=128)

tokenized_dataset2 = dataset2.map(tokenize2, batched=True)
data_collator2 = DataCollatorWithPadding(tokenizer=tokenizer2)

print(tokenized_dataset2)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 4500
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 500
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1000
    })
})


In [16]:
# 加载预训练模型

from transformers import AutoModelForSequenceClassification

model2 = AutoModelForSequenceClassification.from_pretrained(
    model_name2,
    num_labels=num_labels2,
    id2label=id2label2,
    label2id=label2id2,
    ignore_mismatched_sizes=True
)

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [17]:
# 设置评估指标

import evaluate
import numpy as np

metric2 = evaluate.load("accuracy")

def compute_metrics2(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric2.compute(predictions=predictions, references=labels)

In [18]:
# 训练配置

from transformers import TrainingArguments, Trainer

training_args2 = TrainingArguments(
    output_dir="./finbert-topic-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    logging_dir="./logs2",
    report_to="none"
)

trainer2 = Trainer(
    model=model2,
    args=training_args2,
    train_dataset=tokenized_dataset2["train"],
    eval_dataset=tokenized_dataset2["validation"],
    compute_metrics=compute_metrics2,
    data_collator=data_collator2,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [19]:
# 开始训练

trainer2.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.005994,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=141, training_loss=0.043842988656767716, metrics={'train_runtime': 26.3876, 'train_samples_per_second': 170.535, 'train_steps_per_second': 5.343, 'total_flos': 188257602624960.0, 'train_loss': 0.043842988656767716, 'epoch': 1.0})

In [20]:
# 测试集评估

import time

start = time.time()
results2 = trainer2.evaluate(tokenized_dataset2["test"])
end = time.time()

print(f"Test Accuracy: {results2['eval_accuracy']:.4f}")
print(f"Runtime: {end - start:.2f} seconds")

Test Accuracy: 0.8990
Runtime: 1.35 seconds


In [21]:
repo_name_topic = "Nora0211/finbert-topic-finetuned"

# 保存模型和分词器
trainer2.save_model("./finbert-topic-finetuned")
tokenizer2.save_pretrained("./finbert-topic-finetuned")   # 变量名是 tokenizer

# 重新加载
from transformers import AutoModelForSequenceClassification, AutoTokenizer
model2 = AutoModelForSequenceClassification.from_pretrained("./finbert-topic-finetuned")
tokenizer2 = AutoTokenizer.from_pretrained("./finbert-topic-finetuned")

# 上传
model2.push_to_hub(repo_name_topic)
tokenizer2.push_to_hub(repo_name_topic)

print("Pipeline 2 模型上传完成（旧仓库已删除）")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...24sdtka/model.safetensors:  33%|###2      |  144MB /  439MB            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Pipeline 2 模型上传完成（旧仓库已删除）


In [22]:
from huggingface_hub import list_models, model_info

# 检查模型是否存在
try:
    info = model_info("Nora0211/finbert-sentiment-finetuned")
    print(f"✅ 模型存在，最后修改时间：{info.lastModified}")
    print(f"   文件列表：{info.siblings[:5]}")  # 打印前5个文件
except Exception as e:
    print(f"❌ 模型不存在或无法访问：{e}")

✅ 模型存在，最后修改时间：2026-05-26 13:37:38+00:00
   文件列表：[RepoSibling(rfilename='.gitattributes', size=None, blob_id=None, lfs=None), RepoSibling(rfilename='README.md', size=None, blob_id=None, lfs=None), RepoSibling(rfilename='config.json', size=None, blob_id=None, lfs=None), RepoSibling(rfilename='model.safetensors', size=None, blob_id=None, lfs=None), RepoSibling(rfilename='tokenizer.json', size=None, blob_id=None, lfs=None)]
